In [1]:
import datetime
import logging  # 追加：ログ制御用
import warnings  # 追加：警告非表示用
import pandas as pd
import yfinance as yf
from IPython.display import display, HTML

# 警告メッセージ（FutureWarningなど）を非表示にする
warnings.filterwarnings("ignore")

# yfinanceの内部エラー出力を非表示にする
yf_logger = logging.getLogger('yfinance')
yf_logger.setLevel(logging.CRITICAL)

# Define the lookback periods (in business days) for performance calculation.
LOOKBACK_DAYS = {
    "1 Day (%)": -2,
    "3 Days (%)": -4,
    "1 Week (%)": -6,
    "1 Month (%)": -21,
    "6 Months (%)": -126,
    "1 Year (%)": -252,
}

# =====================================================================
# DATA PROCESSING FUNCTION (Supports Market Segmentation)
# =====================================================================
def show_market_performances(market_dict):
    today = datetime.date.today()
    start_date = today - datetime.timedelta(days=60)
    
    # Format numerical values to 2 decimal places
    format_config = {"Latest Price": "{:,.2f}"}
    for column_name in LOOKBACK_DAYS.keys():
        format_config[column_name] = "{:+.2f}%"
        
    # ハイライト用の条件関数
    def highlight_extreme_values(val):
        if pd.isna(val):
            return ''
        if val >= 10:
            return 'background-color: #ffcccc; color: #cc0000; font-weight: bold;'
        elif val <= -10:
            return 'background-color: #cce5ff; color: #004085; font-weight: bold;'
        return ''
        
    # Process each market section
    for market_name, ticker_dict in market_dict.items():
        records = []
        
        # Display market header
        display(HTML(f"<h3 style='margin-top: 20px; color: #333;'>■ {market_name} Market</h3>"))
        
        for name, ticker in ticker_dict.items():
            try:
                # 【修正】 ignore_tz=True を追加してタイムゾーン警告を抑制
                df = yf.download(ticker, start=start_date, end=today, progress=False, ignore_tz=True)
                
                if df.empty:
                    # 元のコードにあったWarning表示も邪魔ならコメントアウトしてください
                    # print(f"[Warning] No data found for: {name} ({ticker})")
                    continue
                    
                prices = df["Close"].squeeze()
                latest_price = prices.iloc[-1]
                
                row = {
                    "Name": name,
                    "Ticker": ticker,
                    "Latest Price": latest_price
                }
                
                for label, index in LOOKBACK_DAYS.items():
                    if len(prices) >= abs(index):
                        past_price = prices.iloc[index]
                    else:
                        past_price = prices.iloc[0]
                    
                    row[label] = ((latest_price - past_price) / past_price) * 100
                    
                records.append(row)
                
            except Exception as e:
                # エラー時のprintも非表示にしたい場合はコメントアウトしてください
                pass
        
        if records:
            df_performance = pd.DataFrame(records)
            
            # .map を使用
            styled_df = (df_performance.style
                         .format(format_config)
                         .map(highlight_extreme_values, subset=list(LOOKBACK_DAYS.keys()))
                         .hide(axis="index"))
            
            display(styled_df)
        else:
            print("No data available for this market.")

In [2]:
# =====================================================================
# CONFIGURATION & EXECUTION (Segmented by Market)
# =====================================================================
MARKET_TICKERS = {
    "United States": {
        "S&P 500": "^GSPC",
        "NASDAQ Composite": "^IXIC",
        "Apple": "AAPL",
        "NVIDIA": "NVDA",
        "Microsoft": "MSFT",
    },
    "Japan": {
        "Nikkei 225": "^N225",
        "Toyota Motor": "7203.T",
        "Sony Group": "6758.T",
        "Mitsubishi UFJ Financial": "8306.T",
    },
    "Singapore": {
        "STI Index": "^STI",
        "DBS Group": "D05.SI",
        "UOB": "U11.SI",
        "Singtel": "Z74.SI",
        "CapitaLand Ascendas REIT": "A17U.SI",
    },
    "Japan Stock": {
        "小松製作所 (6301)": "6301.T",
        "ニデック (6594)": "6594.T",
        "未来工業 (7931)": "7931.T",
        "東部ネットワーク (9036)": "9036.T",
        "ニトリHD (9843)": "9843.T",
        "MRK HD (9980)": "9980.T",
    },
    "US Stock (ETF)": {
        "iシェアーズ コア米国総合債券ETF (AGG)": "AGG"
    }
}

# Run and display the segmented tables
show_market_performances(MARKET_TICKERS)

Name,Ticker,Latest Price,1 Day (%),3 Days (%),1 Week (%),1 Month (%),6 Months (%),1 Year (%)
S&P 500,^GSPC,"7,482.71",-0.28%,-0.01%,-0.22%,+1.04%,+0.94%,+0.94%
NASDAQ Composite,^IXIC,"25,870.65",+0.20%,+0.15%,-1.31%,-0.23%,-1.54%,-1.54%
Apple,AAPL,313.39,+0.88%,+1.54%,+8.30%,+3.93%,+7.08%,+7.08%
NVIDIA,NVDA,204.12,+3.65%,+4.77%,+2.01%,-2.17%,-6.98%,-6.98%
Microsoft,MSFT,383.34,-1.41%,-1.83%,+2.77%,-6.90%,-6.90%,-6.90%


Name,Ticker,Latest Price,1 Day (%),3 Days (%),1 Week (%),1 Month (%),6 Months (%),1 Year (%)
Nikkei 225,^N225,"66,819.05",-2.11%,-4.19%,-5.19%,+4.11%,+7.05%,+7.05%
Toyota Motor,7203.T,"2,889.00",-1.93%,+2.16%,+6.04%,+2.67%,+0.66%,+0.66%
Sony Group,6758.T,"3,468.00",-0.97%,+2.60%,+6.71%,+2.45%,+2.85%,+2.85%
Mitsubishi UFJ Financial,8306.T,"3,433.00",-0.38%,+3.22%,+5.73%,+7.85%,+20.18%,+20.18%


Name,Ticker,Latest Price,1 Day (%),3 Days (%),1 Week (%),1 Month (%),6 Months (%),1 Year (%)
STI Index,^STI,"5,369.57",+0.51%,+2.39%,+4.03%,+8.28%,+8.63%,+8.63%
DBS Group,D05.SI,69.10,+0.67%,+3.51%,+5.61%,+11.76%,+17.58%,+17.58%
UOB,U11.SI,43.33,+3.93%,+7.68%,+9.28%,+14.39%,+17.14%,+17.14%
Singtel,Z74.SI,4.37,-0.23%,-2.24%,-1.58%,+2.34%,-8.00%,-8.00%
CapitaLand Ascendas REIT,A17U.SI,2.49,-0.40%,+0.00%,+1.22%,-0.40%,+0.40%,+0.40%


Name,Ticker,Latest Price,1 Day (%),3 Days (%),1 Week (%),1 Month (%),6 Months (%),1 Year (%)
小松製作所 (6301),6301.T,"6,525.00",-2.76%,-1.54%,+4.25%,-0.65%,-1.36%,-1.36%
ニデック (6594),6594.T,"2,583.00",-2.75%,-7.32%,-5.97%,-0.46%,-6.07%,-6.07%
未来工業 (7931),7931.T,"3,195.00",-0.62%,+1.59%,+3.57%,+7.14%,+2.24%,+2.24%
東部ネットワーク (9036),9036.T,"1,270.00",-0.63%,-0.63%,+0.95%,+3.17%,+2.58%,+2.58%
ニトリHD (9843),9843.T,"2,442.50",-0.89%,+4.40%,+5.74%,-9.79%,+3.41%,+3.41%
MRK HD (9980),9980.T,96.00,+0.00%,+2.13%,+3.23%,-1.03%,-3.03%,-3.03%


Name,Ticker,Latest Price,1 Day (%),3 Days (%),1 Week (%),1 Month (%),6 Months (%),1 Year (%)
iシェアーズ コア米国総合債券ETF (AGG),AGG,98.04,-0.17%,-0.58%,-0.62%,+0.20%,-0.56%,-0.56%


In [3]:
MARKET_TICKERS = {
    "Singapore Healthcare": {
        # --- 病院経営・総合クリニック ---
        "Raffles Medical Group (ラッフルズ医療)": "BSL.SI",  # 【修正】R01.SI から変更
        "IHH Healthcare (マウント・エリザベス等経営)": "Q0F.SI",  # 【修正】I01.SI から変更
        "Thomson Medical Group (トムソン医療)": "A50.SI",
        
        # --- 医療不動産（REIT / 病院・介護施設） ---
        "Parkway Life REIT (パークウェイ・ライフ)": "C2PU.SI",
        "First REIT (ファースト・リート)": "AW9U.SI",
        
        # --- 専門医療・歯科クリニックチェーン ---
        "Q&M Dental Group (歯科大手チェーン)": "QC7.SI",
        "Alliance Healthcare Group (総合診療・クリニック)": "MIJ.SI",  # 【修正】1D8から変更
        "Medinex Limited (医療サポート・クリニック運営)": "OTX.SI",
        
        # --- 医療用消耗品・製薬 ---
        "Medtecs International (医療用消耗品)": "546.SI",
        "Hyphens Pharma (医薬品・皮膚科製品)": "1J5.SI",
        "Riverstone": "AP4.SI"
    }
}
show_market_performances(MARKET_TICKERS)

Name,Ticker,Latest Price,1 Day (%),3 Days (%),1 Week (%),1 Month (%),6 Months (%),1 Year (%)
Raffles Medical Group (ラッフルズ医療),BSL.SI,0.92,-1.61%,-1.61%,+0.55%,-2.66%,-4.69%,-4.69%
IHH Healthcare (マウント・エリザベス等経営),Q0F.SI,2.85,+0.00%,+3.26%,+4.01%,+3.64%,-0.70%,-0.70%
Thomson Medical Group (トムソン医療),A50.SI,0.05,-1.85%,+0.00%,+0.00%,-1.85%,-7.02%,-7.02%
Parkway Life REIT (パークウェイ・ライフ),C2PU.SI,4.11,+0.49%,+0.24%,+1.73%,+3.53%,+3.01%,+3.01%
First REIT (ファースト・リート),AW9U.SI,0.22,-2.17%,-2.17%,+0.00%,+0.00%,-6.25%,-6.25%
Q&M Dental Group (歯科大手チェーン),QC7.SI,0.56,+2.75%,+1.82%,+1.82%,-3.45%,-5.08%,-5.08%
Alliance Healthcare Group (総合診療・クリニック),MIJ.SI,0.16,+0.00%,+0.00%,-3.07%,+0.00%,-3.07%,-3.07%
Medinex Limited (医療サポート・クリニック運営),OTX.SI,0.23,+0.00%,+0.00%,+4.55%,+9.52%,+0.00%,+0.00%
Medtecs International (医療用消耗品),546.SI,0.11,+0.00%,-0.87%,-1.72%,-5.79%,-19.15%,-19.15%
Hyphens Pharma (医薬品・皮膚科製品),1J5.SI,0.35,+0.00%,+0.00%,+0.00%,+7.69%,+4.48%,+4.48%


In [4]:
MARKET_TICKERS = {
"Singapore Food & Agribusiness": {
        "Wilmar International (ウィルマー・世界最大級のパーム油)": "F34.SI",
        "Olam Group (オラム・ココア、コーヒー、ナッツ等大手)": "VC2.SI",
        "Riverstone": "AP4.SI",
        "ヤンジジャン・シップビルディング(BS6)": "BS6.SI",
        "Sembcorp Ind (U96)": "U96.SI",
    }
}
show_market_performances(MARKET_TICKERS)

Name,Ticker,Latest Price,1 Day (%),3 Days (%),1 Week (%),1 Month (%),6 Months (%),1 Year (%)
Wilmar International (ウィルマー・世界最大級のパーム油),F34.SI,3.79,+1.07%,+1.88%,+4.41%,+8.29%,+3.55%,+3.55%
Olam Group (オラム・ココア、コーヒー、ナッツ等大手),VC2.SI,1.19,-1.65%,-2.46%,-0.83%,-5.56%,-2.46%,-2.46%
Riverstone,AP4.SI,0.82,-2.38%,-3.53%,-2.96%,-4.65%,+0.00%,+0.00%
ヤンジジャン・シップビルディング(BS6),BS6.SI,3.43,-1.44%,-3.65%,+0.00%,+1.78%,-12.28%,-12.28%
Sembcorp Ind (U96),U96.SI,5.66,-1.22%,-5.35%,-9.15%,-7.67%,-8.41%,-8.41%
